# Semantic-texture operational preflight

Run all once in a fresh GPU runtime. This notebook authenticates fixed external delivery inputs, invokes only the external bootstrap, and exports one package-produced zero-science quartet to a fresh Drive namespace.

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime, timezone
from hashlib import sha256
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile

from google.colab import drive, userdata

sys.tracebacklimit = 0

EXPECTED_SOURCE_REVISION = "9136303fcc8c72648e6c4fbc365776af4f8dd35c"
EXPECTED_PACKAGE_ARCHIVE_SHA256 = "063a0b5eac30e94bfbe85222bc6c8da81064e51a801563d98bd0ff28144c525e"
EXPECTED_PACKAGE_ARCHIVE_SIZE_BYTES = 6371404
EXPECTED_DELIVERY_MANIFEST_SHA256 = "fb5e5678a4a695af281625baf4d408ea0d1742a2c78bf2ec5bf593dcd36f41b6"
EXPECTED_DELIVERY_MANIFEST_SIZE_BYTES = 665
EXPECTED_PACKAGE_IDENTITY = "82ba552bda3394dac0348e9d07c1bcb9a7c6e221451f9279673a91a3c43c85c4"
EXPECTED_EXTERNAL_BOOTSTRAP_SHA256 = "9efd4299598e7bcc1a2003720f4e93c7be82d762d98496041ed0c4dc37906061"
EXPECTED_EXTERNAL_BOOTSTRAP_SIZE_BYTES = 37990
PACKAGE_ARCHIVE_FILENAME = "semantic-texture-phase-a.zip"
DELIVERY_MANIFEST_FILENAME = PACKAGE_ARCHIVE_FILENAME + ".manifest.json"
EXTERNAL_BOOTSTRAP_FILENAME = "semantic_texture_operational_preflight_bootstrap.py"
DELIVERY_COMPLETION_CHECKSUMS_FILENAME = "SHA256SUMS"

def _sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _copy_create_only_and_verify(
    source: Path,
    destination: Path,
    *,
    expected_sha256: str,
    expected_size_bytes: int,
) -> None:
    if not source.is_file() or destination.exists():
        raise RuntimeError("create-only copy boundary failed")
    if source.stat().st_size != expected_size_bytes or _sha256_file(source) != expected_sha256:
        raise RuntimeError("source artifact identity drifted")
    with source.open("rb") as input_handle, destination.open("xb") as output_handle:
        for block in iter(lambda: input_handle.read(1024 * 1024), b""):
            output_handle.write(block)
        output_handle.flush()
        os.fsync(output_handle.fileno())
    if destination.stat().st_size != expected_size_bytes or _sha256_file(destination) != expected_sha256:
        raise RuntimeError("copied artifact identity drifted")

def _field_names(value: object) -> set[str]:
    if isinstance(value, dict):
        return set(value) | set().union(*(_field_names(item) for item in value.values()), set())
    if isinstance(value, list):
        return set().union(*(_field_names(item) for item in value), set())
    return set()

def _validate_package_delivery(
    artifact_root: Path,
    operational_run_id: str,
) -> tuple[tuple[str, str, str, str], dict[str, tuple[int, str]]]:
    operational_result_name = "semantic_texture_operational_result.json"
    transport_result_name = "semantic_texture_transport_result.json"
    operational_delivery = (artifact_root / operational_result_name).is_file()
    transport_delivery = (artifact_root / transport_result_name).is_file()
    if operational_delivery == transport_delivery:
        raise RuntimeError("operational and transport delivery must be mutually exclusive")
    if operational_delivery:
        result_name = operational_result_name
        archive_name = f"semantic_texture_operational_{operational_run_id}.zip"
        receipt_name = "semantic_texture_operational_receipt.json"
    else:
        result_name = transport_result_name
        archive_name = "semantic_texture_transport_result.zip"
        receipt_name = "semantic_texture_transport_receipt.json"
    artifact_names = (result_name, archive_name, receipt_name, DELIVERY_COMPLETION_CHECKSUMS_FILENAME)
    if {path.name for path in artifact_root.iterdir() if path.is_file()} != set(artifact_names):
        raise RuntimeError("package delivery quartet is incomplete")
    result_path = artifact_root / result_name
    archive_path = artifact_root / archive_name
    receipt_path = artifact_root / receipt_name
    completion_checksums_path = artifact_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME
    result_blob = result_path.read_bytes()
    result = json.loads(result_blob)
    if (
        result.get("status") != "blocked"
        or result.get("aggregate") is not None
        or result.get("science_started") is not False
        or result.get("scientific_unit_count") != 0
        or result.get("formal_tau_created") is not False
        or result.get("candidate_promoted") is not False
        or result.get("scientific_claims_supported") is not False
    ):
        raise RuntimeError("zero-science blocked boundary drifted")
    if operational_delivery:
        if (
            result.get("source_revision") != EXPECTED_SOURCE_REVISION
            or result.get("package_identity") != EXPECTED_PACKAGE_IDENTITY
            or result.get("run_id") != operational_run_id
            or result.get("asset_authority_status") != "identity_blocked"
        ):
            raise RuntimeError("operational result identity drifted")
        if any(
            outcome.get("sanitized_error_message") is not None
            or outcome.get("sanitized_trace_tail") != []
            for outcome in result.get("unit_outcomes", [])
        ):
            raise RuntimeError("operational error persistence drifted")
    elif (
        result.get("source_revision") not in (None, EXPECTED_SOURCE_REVISION)
        or result.get("package_identity") not in (None, EXPECTED_PACKAGE_IDENTITY)
        or "unit_outcomes" in result
    ):
        raise RuntimeError("transport result authority drifted")
    forbidden_fields = {"CEG_WM_ROOT_KEY", "HF_TOKEN", "M", "T", "generation_prompt", "generation_seed", "latent", "m_hf", "m_lf", "mask", "private_state", "root_key", "token"}
    receipt_blob = receipt_path.read_bytes()
    receipt = json.loads(receipt_blob)
    if forbidden_fields & (_field_names(result) | _field_names(receipt)):
        raise RuntimeError("private or generation state crossed persistence boundary")
    with zipfile.ZipFile(archive_path) as archive:
        if archive.namelist() != [result_name] or archive.read(result_name) != result_blob:
            raise RuntimeError("result-only archive boundary drifted")
    if (
        receipt.get("result_filename") != result_name
        or receipt.get("archive_filename") != archive_name
        or receipt.get("result_sha256") != sha256(result_blob).hexdigest()
        or receipt.get("archive_sha256") != _sha256_file(archive_path)
    ):
        raise RuntimeError("external receipt binding drifted")
    completion_lines = completion_checksums_path.read_text(encoding="ascii").splitlines()
    if [line.split("  ", 1)[1] for line in completion_lines] != [result_name, archive_name, receipt_name]:
        raise RuntimeError("package completion checksum roster drifted")
    for line in completion_lines:
        expected_digest, artifact_name = line.split("  ", 1)
        if _sha256_file(artifact_root / artifact_name) != expected_digest:
            raise RuntimeError("package completion checksum drifted")
    persisted_blob = result_blob + archive_path.read_bytes() + receipt_blob + completion_checksums_path.read_bytes()
    forbidden_fragments = [b"/content/", b"traceback"]
    if any(fragment and fragment in persisted_blob for fragment in forbidden_fragments):
        raise RuntimeError("secret, path, or traceback crossed persistence boundary")
    identities = {name: ((artifact_root / name).stat().st_size, _sha256_file(artifact_root / name)) for name in artifact_names}
    return artifact_names, identities


In [ ]:
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    drive.mount("/content/drive")
operational_run_id = "semantic-texture-operational-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
drive_input_root = Path("/content/drive/MyDrive/CEG-WM/semantic_texture_operational_preflight/inputs") / EXPECTED_SOURCE_REVISION
drive_export_root = Path("/content/drive/MyDrive/CEG-WM/semantic_texture_operational_preflight/exports") / EXPECTED_SOURCE_REVISION / operational_run_id
local_execution_root = Path("/content/ceg-wm-semantic-texture-operational") / operational_run_id
if local_execution_root.exists() or drive_export_root.exists():
    raise RuntimeError("fresh local and Drive roots are required; resume, migrate, and overwrite are forbidden")
local_input_root = local_execution_root / "trusted-inputs"
local_input_root.mkdir(parents=True)
local_package_archive = local_input_root / PACKAGE_ARCHIVE_FILENAME
local_delivery_manifest = local_input_root / DELIVERY_MANIFEST_FILENAME
local_external_bootstrap = local_input_root / EXTERNAL_BOOTSTRAP_FILENAME
_copy_create_only_and_verify(
    drive_input_root / PACKAGE_ARCHIVE_FILENAME,
    local_package_archive,
    expected_sha256=EXPECTED_PACKAGE_ARCHIVE_SHA256,
    expected_size_bytes=EXPECTED_PACKAGE_ARCHIVE_SIZE_BYTES,
)
_copy_create_only_and_verify(
    drive_input_root / DELIVERY_MANIFEST_FILENAME,
    local_delivery_manifest,
    expected_sha256=EXPECTED_DELIVERY_MANIFEST_SHA256,
    expected_size_bytes=EXPECTED_DELIVERY_MANIFEST_SIZE_BYTES,
)
_copy_create_only_and_verify(
    drive_input_root / EXTERNAL_BOOTSTRAP_FILENAME,
    local_external_bootstrap,
    expected_sha256=EXPECTED_EXTERNAL_BOOTSTRAP_SHA256,
    expected_size_bytes=EXPECTED_EXTERNAL_BOOTSTRAP_SIZE_BYTES,
)
delivery_manifest = json.loads(local_delivery_manifest.read_text(encoding="utf-8"))
if (
    delivery_manifest.get("source_revision") != EXPECTED_SOURCE_REVISION
    or delivery_manifest.get("package_identity") != EXPECTED_PACKAGE_IDENTITY
    or delivery_manifest.get("archive_sha256") != EXPECTED_PACKAGE_ARCHIVE_SHA256
    or delivery_manifest.get("archive_size_bytes") != EXPECTED_PACKAGE_ARCHIVE_SIZE_BYTES
):
    raise RuntimeError("delivery manifest authority drifted")
hf_token = userdata.get("HF_TOKEN")
root_key = userdata.get("CEG_WM_ROOT_KEY")
if not isinstance(hf_token, str) or not hf_token or not isinstance(root_key, str) or not root_key:
    raise RuntimeError("required Colab Secrets are unavailable")


In [ ]:
extract_root = local_execution_root / "gitless"
operational_output_root = local_execution_root / "operational-delivery"
ephemeral_root = local_execution_root / "tmp"
ephemeral_root.mkdir()
bootstrap_environment = os.environ.copy()
bootstrap_environment.update({
    "HF_TOKEN": hf_token,
    "CEG_WM_ROOT_KEY": root_key,
    "PYTHONDONTWRITEBYTECODE": "1",
    "TMPDIR": str(ephemeral_root),
    "TMP": str(ephemeral_root),
    "TEMP": str(ephemeral_root),
})
bootstrap_command = [
    sys.executable,
    str(local_external_bootstrap),
    "--archive", str(local_package_archive),
    "--manifest", str(local_delivery_manifest),
    "--expected-sha256", EXPECTED_PACKAGE_ARCHIVE_SHA256,
    "--expected-size", str(EXPECTED_PACKAGE_ARCHIVE_SIZE_BYTES),
    "--extract-root", str(extract_root),
    "--entrypoint-args",
    "--execute",
    "--source-revision", EXPECTED_SOURCE_REVISION,
    "--run-id", operational_run_id,
    "--package-identity", EXPECTED_PACKAGE_IDENTITY,
    "--output-root", str(operational_output_root),
]
bootstrap_completed = subprocess.run(
    bootstrap_command,
    check=False,
    capture_output=True,
    env=bootstrap_environment,
)
del bootstrap_environment, hf_token, root_key
transport_output_root = extract_root.with_name(extract_root.name + ".transport")
operational_delivery_exists = operational_output_root.is_dir()
transport_delivery_exists = transport_output_root.is_dir()
if operational_delivery_exists == transport_delivery_exists:
    raise RuntimeError("exactly one package delivery quartet is required")
local_artifact_root = operational_output_root if operational_delivery_exists else transport_output_root
artifact_names, artifact_identities = _validate_package_delivery(
    local_artifact_root,
    operational_run_id,
)
drive_export_root.mkdir(parents=True)
for artifact_name in artifact_names[:3]:
    artifact_size, artifact_sha256 = artifact_identities[artifact_name]
    _copy_create_only_and_verify(
        local_artifact_root / artifact_name,
        drive_export_root / artifact_name,
        expected_sha256=artifact_sha256,
        expected_size_bytes=artifact_size,
    )
local_completion_checksums = local_artifact_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME
drive_completion_checksums = drive_export_root / DELIVERY_COMPLETION_CHECKSUMS_FILENAME
pending_completion_checksums = drive_export_root / ".package-completion-checksums.pending"
completion_checksums_blob = local_completion_checksums.read_bytes()
with pending_completion_checksums.open("xb") as output_handle:
    output_handle.write(completion_checksums_blob)
    output_handle.flush()
    os.fsync(output_handle.fileno())
if pending_completion_checksums.read_bytes() != completion_checksums_blob or drive_completion_checksums.exists():
    raise RuntimeError("completion checksum atomic-copy boundary failed")
os.replace(pending_completion_checksums, drive_completion_checksums)
if drive_completion_checksums.read_bytes() != completion_checksums_blob:
    raise RuntimeError("package completion checksums changed during Drive copy")
for artifact_name in artifact_names:
    artifact_size, artifact_sha256 = artifact_identities[artifact_name]
    drive_artifact = drive_export_root / artifact_name
    if drive_artifact.stat().st_size != artifact_size or _sha256_file(drive_artifact) != artifact_sha256:
        raise RuntimeError("Drive artifact identity drifted")
if {path.name for path in drive_export_root.iterdir()} != set(artifact_names):
    raise RuntimeError("Drive quartet contains an unexpected artifact")
if bootstrap_completed.returncode != 0:
    raise RuntimeError(f"external bootstrap exited {bootstrap_completed.returncode} after complete immutable Drive delivery") from None
raise RuntimeError("external bootstrap unexpectedly returned zero for the blocked operational roster") from None
